# Word To Vector
## 1. Introduction
In the last notebook TF-IDF was used to improve our results from a standard OGBN database. TF-IDF uses two calculations TF and IDF, then merges them by multiplying TF and IDF giving a final score of weights. Term Frequency calculates how many times a term appears in a document and averages. You count how many times a specific word appears in the document and divide by the total number of words in the document. Inverse Document Frequency—is the log of the frequency: "how many papers use that term or how many times does it appear in different documents?" But it is the inverse so the number of total papers is divided by the number of papers where the word appears. This penalizes words that appear often and promotes words that are distinct, creating clusters of word documents by shared vocabulary. The problem is the same word is often used differently depending on the context, that is, words carry semantic meaning. TF-IDF is just a bag of words and doesn't capture these meanings. This means the word apple in the sentence: "An apple fell of a tree" is treated the same as "Apple products are typically priced more." How do we fix this?

Word2Vec assumes that words that appear in similar contexts probably have similar meanings. The model is trained on massive amounts of text such as Google News with 100 billion words. It learns that **words that appear near each are related** and then predicts surround words from center words (skip gram), or center words from surrounding words (CBOW).

**Example training:**
```
Text: "The neural network processes information efficiently"

Word2Vec learns:
- "neural" often appears near: network, neurons, brain, cortex
- "network" often appears near: neural, connections, nodes, layer
- Therefore: "neural" and "network" should be CLOSE in vector space
```

Each word becomes a 300-dimensional point in space where similar words cluster together and relationships are encoded as directions. There is a famous example that illustrates that ability to do math with meanings:

Famous examples:
```
king - man + woman ≈ queen
Paris - France + Italy ≈ Rome
walking - walk + swim ≈ swimming
```

### 1.2 Why This Matters for ArXiv

#### TF-IDF strengths on ArXiv:
- Captures domain-specific vocabulary ("transformers", "convolution", "eigenvalues")
- Distinguishes it by technical term usage
- Perfect match: vocabulary learned from your exact data

#### Word2Vec strengths on ArXiv:
- **Semantic generalization**: Understands "ML" and "machine learning" are related
- **Synonym handling**: "optimize" and "minimize" are treated similarly
- **Concept capture**: Words like "gradient," "descent," "optimization" cluster together
- **Transfer learning**: Brings general language understanding from Google News

#### Word2Vec weaknesses on ArXiv:
- **Domain mismatch**: Google News doesn't have "Riemannian manifolds" or "LSTM"
- **OOV rate**: Many scientific terms won't exist in the pretrained vocabulary
- **Lost specificity**: Can't distinguish technical terms it hasn't seen

<br>

## 2. Aggregation
### 2.1 What we want to solve.
<b>TF-IDF:</b><br>
<b>Abstract:</b> "neural networks optimize loss functions"<br>
<b>Output:</b><br>
&emsp;&emsp;[0, 0, 0.5, 0, 0.3, 0, 0, 0.8, 0, ...] (one vector directly)<br>

<b>Word2Vec:</b><br>
<b>Abstract:</b> "neural networks optimize loss functions"<br>
<b>Step 1:</b><br>
        &emsp;&emsp;"neural" → [0.2, -0.5, 0.1, ...] (300-dim)<br>
        &emsp;&emsp;"networks" → [0.3, -0.4, 0.2, ...] (300-dim)<br>
        &emsp;&emsp;"optimize" → [-0.1, 0.6, 0.3, ...] (300-dim)<br>
        &emsp;&emsp;"loss" → [0.4, -0.2, 0.1, ...] (300-dim)<br>
        &emsp;&emsp;"functions" → [0.1, 0.3, -0.2, ...] (300-dim)<br>

### 2.2 How do we solve it?
Word2Vec creates a 300-dimensional vector for each word in the document to encode semantic meaning. However, for training, we need a single vector per document/example. Then how do we create a single vector per document from the 300-dimensional vector for each word in the abstract/text? The answer is aggregation, and there exist multiple strategies for aggregating differently.
1. <b>Mean Pooling (Average):</b> The documents meaning is the average of the word meanings. This method is best for general semantic similarity when all words roughly equal importance.<br>
    - <b>Pros:</b>
        - Simple and Interpretable
        - All words contribute equally
        - Smooth representation
    - <b>Cons:</b>
        - Unless removed, stop words dilute important words
        - Loses word order completely
        - Long documents might lose specificity

<br>

2. <b>Max Pooling (Element-wise Maximum):</b> Take the strongest signal in each dimension. These could represent key terms, therefore, use this method when key terms matter more than the overall topic.
    - <b>Pros:</b>
        - Captures the most notable or important words
        - Less diluted by stop words
        - Each dimension represents the most extreme dimension or the loudest signal
    - <b>Cons:</b>
        - Loses magnitude information
        - Can be noisy if one word has extreme values

<br>

3. <b>Weighted Average (TF-IDF weights):</b> TF-IDF clusters documents by shared vocabulary representing important distinguishable words in those clusters. By default, each word contributes equally, but weighted average aggregation enables important words to contribute more to less important words. This method captures both semantic meaning and term distinctiveness.
    - <b>Pros:</b>
        - Combines TF-IDF's discriminative power with Word2Vec's semantics
        - Downweights common words
        - Theory: best of both worlds
    - <b>Cons:</b>
        - More complex—need to compute TF-IDF first
        - Two-stage process
        - Coupling two different philosophies

<br>

4. <b>Concatenate First/Last N:</b> This method is best for structured text with meaningful positions, abstract structure matters—introduction vs. conclusion: `doc_vector = concat(avg(first_10_words), avg(last_10_words))`.
    - <b>Pros:</b>
        - Captures positional information
        - Scientific abstracts have structure (background → methods → results)
        - Richer representation (600-dim instead of 300-dim)
    - <b>Cons:</b>
        - Fixed window size arbitrary
        - Assumes abstracts follow convention
        - Doubles dimensionality
<br>

## 3. Imports and Project Initialization
### 3.1 Imports

In [3]:
import pandas as pd


input_path = '../data/processed/arxiv_text.parquet'

In [15]:
print(f"Loading data from {input_path}")
data = pd.read_parquet(input_path)

print(f"The shape of the input data is {data.shape}")
print(f"The columns or features are {data.columns.to_list()}")
print(f"First 5 rows:")
data.head()

Loading data from ../data/processed/arxiv_text.parquet
The shape of the input data is (169343, 7)
The columns or features are ['paper id', 'node idx', 'split', 'label', 'arxiv category', 'year', 'text']
First 5 rows:


,paper id,node idx,split,label,arxiv category,year,text
0,630234,104447,train,6,arxiv cs hc,2011,spreadsheets on the move an evaluation of mobi...
1,803423,15858,train,16,arxiv cs cv,2014,multi view metric learning for multi view vide...
2,1102481,107156,train,5,arxiv cs dc,2013,big data analytics in future internet of thing...
3,1532644,141536,train,24,arxiv cs lg,2014,machine learner for automated reasoning 0 4 an...
4,1810480,82077,train,4,arxiv cs cr,2011,cryptographic hardening of d sequences this pa...
